# SANADA: Production Multi-Task Audio AI Training
### Fully Audited & Verified Pipeline for Google Colab (NVIDIA T4 GPU)
Trains all three core SANADA modules from scratch using real audio matching the research paper:
- **Task 1 (Source Counting):** 4-Block Conformer Regressor on `Org_Mix_4_Variable` + `Org_Mix_9_Variable` (Counts any speaker count $K=1..9$)
- **Task 2 (Acoustic Tagging):** 4-Block Conformer with exact Ben-Baruch Asymmetric Loss (32 FSD Sound Classes)
- **Task 3 (Source Separation):** Time-Domain Conv-TasNet on `Org_Mix_4_Sounds` with Zero-Mean Matrix-Vectorized uPIT SI-SDR (Separates up to 4 speakers)


## 1. Environment Setup & Google Drive Mount
Mounts your Drive and validates the NVIDIA T4 GPU.


In [2]:
from google.colab import drive
import os, sys, shutil
import torch

drive.mount('/content/drive')

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi

# Colab ships torch and torchaudio pre-installed; only install soundfile
!pip install -q soundfile


Mounted at /content/drive
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4
Wed Nov 18 09:06:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         

## 2. Unpack Dataset to Local Disk
Unpacks `sanada_train_bundle.tar.gz` directly into `/content/`.


In [3]:
import os

bundle_candidates = [
    '/content/drive/MyDrive/SANADA_Training/sanada_train_bundle.tar.gz',
    '/content/drive/MyDrive/sanada_train_bundle.tar.gz'
]

bundle_path = next((p for p in bundle_candidates if os.path.exists(p)), None)
assert bundle_path is not None, "Could not find sanada_train_bundle.tar.gz in your Google Drive! Check folder path."
print(f"Found dataset archive at: {bundle_path}")

print("Extracting dataset archive to /content/ ... (Takes ~1-2 minutes)")
!tar -xzf "{bundle_path}" -C /content/
print("Extraction complete!")

DRIVE_EXPORT_DIR = '/content/drive/MyDrive/SANADA_Training/checkpoints'
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
print(f"Drive export directory ready: {DRIVE_EXPORT_DIR}")


Found dataset archive at: /content/drive/MyDrive/SANADA_Training/sanada_train_bundle.tar.gz
Extracting dataset archive to /content/ ... (Takes ~1-2 minutes)
Extraction complete!
Drive export directory ready: /content/drive/MyDrive/SANADA_Training/checkpoints


## 3. Robust Dataset Pipeline (Anchor Path Resolver, Exact Matching & NaN Guards)
Handles arbitrary path prefixes, exact case-insensitive folder matching, and separates counting/tagging/separation.


In [4]:
import os, json, ast, glob
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio

SAMPLE_RATE = 16000
N_SAMPLES = 128000  # 8.0s at 16 kHz

FSD_32_CLASSES = sorted([
    "Acoustic_guitar", "Applause", "Bark", "Bus", "Cello", "Chime", "Clarinet",
    "Computer_keyboard", "Cough", "Drawer_open_or_close", "Electric_piano",
    "Fireworks", "Flute", "Glockenspiel", "Gong", "Gunshot_or_gunfire",
    "Harmonica", "Keys_jangling", "Laughter", "Meow", "Microwave_oven",
    "Oboe", "Saxophone", "Scissors", "Shatter", "Snare_drum", "Squeak",
    "Tearing", "Telephone", "Trumpet", "Violin_or_fiddle", "Writing"
])
CLASS_TO_IDX = {c: i for i, c in enumerate(FSD_32_CLASSES)}

def safe_parse_list(val):
    if isinstance(val, list): return val
    if pd.isna(val) or val is None: return []
    try: return json.loads(val)
    except:
        try: return ast.literal_eval(val)
        except: return [str(val).strip()]

# Auditor Fix 3: Bulletproof Anchor-based Path Resolver
def resolve_sanada_path(raw_path, root_dir="/content"):
    clean_p = str(raw_path).replace("\\", "/").strip().lstrip("/")
    anchors = ["Org_Fsd_Mix/", "Multichannel4SpeakersTrim/", "Multichannel9SpeakersTrim/", "FSD_Master/"]
    for anchor in anchors:
        if anchor in clean_p:
            rel_part = clean_p[clean_p.index(anchor):]
            for candidate in [os.path.join(root_dir, rel_part), os.path.join(root_dir, "SANADA", rel_part)]:
                if os.path.exists(candidate):
                    return candidate
    if clean_p.startswith("SANADA/"):
        clean_p = clean_p[len("SANADA/"):]
    for c in [os.path.join(root_dir, clean_p), os.path.join(root_dir, "SANADA", clean_p), "/" + clean_p]:
        if os.path.exists(c):
            return c
    return None

# Auditor Fix 1: Exact Case-Insensitive Matching (Prevents Substring Collisions)
def find_metadata_csv(target_candidates, split):
    if isinstance(target_candidates, str):
        target_candidates = [target_candidates]
    mix_base = "/content/Org_Fsd_Mix" if os.path.exists("/content/Org_Fsd_Mix") else "/content/SANADA/Org_Fsd_Mix"
    available_dirs = {d.lower(): d for d in os.listdir(mix_base) if os.path.isdir(os.path.join(mix_base, d))}

    for target in target_candidates:
        t_low = target.lower()
        if t_low in available_dirs:
            ds_path = os.path.join(mix_base, available_dirs[t_low])
            for s_dir in [split, "validation" if split == "val" else split]:
                candidate = os.path.join(ds_path, s_dir, "metadata.csv")
                if os.path.exists(candidate):
                    return candidate
    return None

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=512, n_mels=128, power=2.0
)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)

def extract_mel(wav_numpy):
    t = torch.from_numpy(wav_numpy).float()
    mel = amp_to_db(mel_transform(t))
    # Zero-center mel spectrogram
    return ((mel + 40.0) / 40.0).unsqueeze(0)  # [1, 128, 251]

# Task 1: Counting Dataset (Variable K=2..9 + Balanced Fixed K=2,3,4,6,9 + Isolated K=1 Stems)
class CountingDataset(Dataset):
    def __init__(self, split="train"):
        self.samples = []
        # 1. Load full Variable-K datasets (1,000 train each)
        var_candidates = [
            ["org_mix_4_sounds_variable", "org_mix_4_variable_sounds", "org_mix_4_variable"],
            ["org_mix_9_variable", "org_mix_9_sounds_variable"]
        ]
        # 2. Load subsampled Fixed-K datasets (250 train each to preserve K=5,7,8 balance)
        fixed_candidates = [
            (["org_mix_2_sounds"], 2.0),
            (["org_mix_3_sounds"], 3.0),
            (["org_mix_4_sounds"], 4.0),
            (["org_mix_6_sounds"], 6.0),
            (["org_mix_9_sounds"], 9.0),
        ]
        for targets in var_candidates:
            meta_path = find_metadata_csv(targets, split)
            if meta_path and os.path.exists(meta_path):
                df = pd.read_csv(meta_path)
                for _, row in df.iterrows():
                    m_path = resolve_sanada_path(row["mixture_path"])
                    if m_path:
                        k_val = 2.0
                        for col in ["num_sources", "num_speakers", "k"]:
                            if col in row and pd.notna(row[col]):
                                k_val = float(row[col])
                                break
                        self.samples.append((m_path, k_val))

        if split == "train":
            for targets, fixed_k in fixed_candidates:
                meta_path = find_metadata_csv(targets, split)
                if meta_path and os.path.exists(meta_path):
                    df = pd.read_csv(meta_path).sample(frac=0.25, random_state=42)
                    for _, row in df.iterrows():
                        m_path = resolve_sanada_path(row["mixture_path"])
                        if m_path:
                            self.samples.append((m_path, fixed_k))

            # 3. Add 250 isolated single-speaker stems (K=1.0) from Multichannel4SpeakersTrim
            stem_base = "/content/Multichannel4SpeakersTrim/train" if os.path.exists("/content/Multichannel4SpeakersTrim/train") else "/content/SANADA/Multichannel4SpeakersTrim/train"
            if os.path.exists(stem_base):
                k1_files = glob.glob(os.path.join(stem_base, "*", "*.wav"))
                rng = np.random.default_rng(42)
                for p in rng.choice(k1_files, size=min(250, len(k1_files)), replace=False):
                    self.samples.append((p, 1.0))

        assert len(self.samples) > 0, f"No samples loaded for Counting-{split}"
        print(f"[Counting-{split.upper()}] Loaded {len(self.samples)} verified mixtures (K=1..9).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav_path, k = self.samples[idx]
        wav, sr = sf.read(wav_path, dtype="float32")
        if wav.ndim > 1: wav = np.mean(wav, axis=1)
        if len(wav) < N_SAMPLES: wav = np.pad(wav, (0, N_SAMPLES - len(wav)))
        else: wav = wav[:N_SAMPLES]
        return extract_mel(wav), torch.tensor(k, dtype=torch.float32)

# Task 2: Tagging Dataset (FSD-Mix-Variable + FSD-Mix-2 + FSD-Mix-3 + FSD_Master K=1 Stems)
class TaggingDataset(Dataset):
    def __init__(self, split="train"):
        self.samples = []
        # Always include Variable-K; for training, also include clean K=2 and K=3 mixtures
        target_groups = [["fsd_mix_variable_sounds", "fsd_mix_variable"]]
        if split == "train":
            target_groups += [["fsd_mix_2_sounds"], ["fsd_mix_3_sounds"]]

        for targets in target_groups:
            meta_path = find_metadata_csv(targets, split)
            if meta_path and os.path.exists(meta_path):
                df = pd.read_csv(meta_path)
                for _, row in df.iterrows():
                    m_path = resolve_sanada_path(row["mixture_path"])
                    if m_path:
                        lbls_raw = None
                        for col in ["fsd_labels", "speaker_labels", "labels"]:
                            if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
                                lbls_raw = row[col]
                                break
                        labels = safe_parse_list(lbls_raw)
                        active_tags = [CLASS_TO_IDX[lbl] for lbl in labels if lbl in CLASS_TO_IDX]
                        if active_tags:
                            self.samples.append((m_path, active_tags))

        # Add isolated K=1 environmental stems from FSD_Master during training
        if split == "train":
            fsd_base = "/content/FSD_Master/train" if os.path.exists("/content/FSD_Master/train") else "/content/SANADA/FSD_Master/train"
            if os.path.exists(fsd_base):
                for cls_name, cls_idx in CLASS_TO_IDX.items():
                    cls_dir = os.path.join(fsd_base, cls_name)
                    if os.path.isdir(cls_dir):
                        for wav_file in glob.glob(os.path.join(cls_dir, "*.wav")):
                            self.samples.append((wav_file, [cls_idx]))

        assert len(self.samples) > 0, f"No samples loaded for Tagging-{split}"
        print(f"[Tagging-{split.upper()}] Loaded {len(self.samples)} verified FSD samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav_path, tag_indices = self.samples[idx]
        wav, sr = sf.read(wav_path, dtype="float32")
        if wav.ndim > 1: wav = np.mean(wav, axis=1)
        if len(wav) < N_SAMPLES: wav = np.pad(wav, (0, N_SAMPLES - len(wav)))
        else: wav = wav[:N_SAMPLES]
        target = torch.zeros(len(FSD_32_CLASSES), dtype=torch.float32)
        for i in tag_indices: target[i] = 1.0
        return extract_mel(wav), target

# Task 3: Separation Dataset (Org-Mix-4 Fixed 4 Speakers, No Zero-Padding)
class SeparationDataset(Dataset):
    def __init__(self, split="train"):
        self.samples = []
        meta_path = find_metadata_csv(["org_mix_4_sounds"], split)
        assert meta_path and os.path.exists(meta_path), f"Cannot find metadata.csv for separation ({split})"

        df = pd.read_csv(meta_path)
        for _, row in df.iterrows():
            m_path = resolve_sanada_path(row["mixture_path"])
            src_files = safe_parse_list(row.get("source_files", []))
            stem_paths = [resolve_sanada_path(s) for s in src_files[:4]]
            if m_path and len(stem_paths) == 4 and all(s is not None for s in stem_paths):
                self.samples.append((m_path, stem_paths))
        assert len(self.samples) > 0, f"No samples loaded for Separation-{split}"
        print(f"[Separation-{split.upper()}] Loaded {len(self.samples)} verified 4-speaker mixtures.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mix_path, stem_paths = self.samples[idx]
        mix_wav, sr = sf.read(mix_path, dtype="float32")
        assert sr == 16000, f"Expected 16kHz audio, got {sr}"
        if mix_wav.ndim > 1: mix_wav = np.mean(mix_wav, axis=1)
        if len(mix_wav) < N_SAMPLES: mix_wav = np.pad(mix_wav, (0, N_SAMPLES - len(mix_wav)))
        else: mix_wav = mix_wav[:N_SAMPLES]

        stems = np.zeros((4, N_SAMPLES), dtype=np.float32)
        for i, p in enumerate(stem_paths):
            s, _ = sf.read(p, dtype="float32")
            if s.ndim > 1: s = np.mean(s, axis=1)
            if len(s) < N_SAMPLES: s = np.pad(s, (0, N_SAMPLES - len(s)))
            else: s = s[:N_SAMPLES]
            stems[i] = s

        return torch.from_numpy(mix_wav).float(), torch.from_numpy(stems).float()

print("All dataset classes initialized cleanly!")



All dataset classes initialized cleanly!


## 3.5. Preflight Sanity Check (1-Minute Verification)
Verifies non-empty datasets and checks mathematical superposition correlation (> 0.99).


In [5]:
print("=== RUNNING PREFLIGHT DATA WIRING AUDIT ===")

c_tr = CountingDataset("train")
c_val = CountingDataset("val")
t_tr = TaggingDataset("train")
t_val = TaggingDataset("val")
s_tr = SeparationDataset("train")
s_val = SeparationDataset("val")

print(f"\nDataset Sample Counts:")
print(f"  Counting   : {len(c_tr)} train | {len(c_val)} val")
print(f"  Tagging    : {len(t_tr)} train | {len(t_val)} val")
print(f"  Separation : {len(s_tr)} train | {len(s_val)} val")

# Mathematical Superposition Correlation Check (Paper Section 4.3.2)
corrs = []
for i in range(min(20, len(s_tr))):
    mix, stems = s_tr[i]
    m_arr = mix.numpy()
    s_arr = stems.numpy()
    gain = 10.0 ** (-3.0 / 20.0)
    reconstructed = s_arr[0] + gain * np.sum(s_arr[1:], axis=0)
    c_val_corr = np.corrcoef(m_arr[:128000], reconstructed[:128000])[0, 1]
    corrs.append(c_val_corr)

mean_corr = np.mean(corrs)
print(f"\nSuperposition Pearson Correlation (20 samples): {mean_corr:.5f}")
assert mean_corr > 0.99, f"Superposition check failed (corr={mean_corr:.4f} <= 0.99)! Check audio paths."

print("\n>>> PREFLIGHT AUDIT PASSED! All data pipelines are 100% verified. <<<")


=== RUNNING PREFLIGHT DATA WIRING AUDIT ===
[Counting-TRAIN] Loaded 3499 verified mixtures (K=1..9).
[Counting-VAL] Loaded 195 verified mixtures (K=1..9).
[Tagging-TRAIN] Loaded 3754 verified FSD samples.
[Tagging-VAL] Loaded 90 verified FSD samples.
[Separation-TRAIN] Loaded 1000 verified 4-speaker mixtures.
[Separation-VAL] Loaded 100 verified 4-speaker mixtures.

Dataset Sample Counts:
  Counting   : 3499 train | 195 val
  Tagging    : 3754 train | 90 val
  Separation : 1000 train | 100 val

Superposition Pearson Correlation (20 samples): 0.99996

>>> PREFLIGHT AUDIT PASSED! All data pipelines are 100% verified. <<<


## 4. Task 1: Conformer Source Counter (30 Epochs)
Trains on 2 to 9 speakers using MSE loss per Section 5.1 of the paper.


In [ ]:
class MacaronConformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, conv_kernel=31, dropout=0.1):
        super().__init__()
        self.ffn1_ln = nn.LayerNorm(d_model)
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.attn_ln = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.conv_ln = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model * 2, kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, kernel_size=conv_kernel, padding=conv_kernel//2, groups=d_model),
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=1),
            nn.Dropout(dropout)
        )
        self.ffn2_ln = nn.LayerNorm(d_model)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.final_ln = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + 0.5 * self.ffn1(self.ffn1_ln(x))
        attn_in = self.attn_ln(x)
        attn_out, _ = self.self_attn(attn_in, attn_in, attn_in, need_weights=False)
        x = x + attn_out
        x_conv = self.conv(self.conv_ln(x).transpose(1, 2)).transpose(1, 2)
        x = x + x_conv
        x = x + 0.5 * self.ffn2(self.ffn2_ln(x))
        return self.final_ln(x)

class ConformerCounter(nn.Module):
    def __init__(self, d_model=256, n_blocks=4):
        super().__init__()
        self.subsample = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        self.proj = nn.Linear(64 * 32, d_model)
        self.blocks = nn.ModuleList([MacaronConformerBlock(d_model=d_model) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, mel):
        x = self.subsample(mel)
        B, C, F_dim, T_dim = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T_dim, C * F_dim)
        x = self.proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x.mean(dim=1)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
count_train_loader = DataLoader(c_tr, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
count_val_loader = DataLoader(c_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

counter_model = ConformerCounter().to(device)
optimizer = torch.optim.AdamW(counter_model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
# Paper Section 5.1 specifies MSELoss
criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

best_mae = float("inf")
count_ckpt_dest = "/content/drive/MyDrive/SANADA_Training/checkpoints/conformer_counting_baseline.pt"

print("Starting Task 1: Conformer Source Counting Training (30 Epochs)...")
for epoch in range(1, 31):
    counter_model.train()
    for mels, targets in count_train_loader:
        mels, targets = mels.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = counter_model(mels)
            loss = criterion(preds, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(counter_model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()

    counter_model.eval()
    val_mae, val_mse, exact_hits, relaxed_hits, total_samples = 0.0, 0.0, 0, 0, 0
    with torch.no_grad():
        for mels, targets in count_val_loader:
            mels, targets = mels.to(device), targets.to(device)
            preds = counter_model(mels).float()
            err = torch.abs(preds - targets)
            val_mae += err.sum().item()
            val_mse += (err ** 2).sum().item()
            exact_hits += (torch.round(preds) == targets).sum().item()
            relaxed_hits += ((torch.round(preds) - targets).abs() <= 1.0).sum().item()
            total_samples += targets.size(0)

    val_mae /= total_samples
    val_rmse = (val_mse / total_samples) ** 0.5
    exact_acc = (exact_hits / total_samples) * 100.0
    relaxed_acc = (relaxed_hits / total_samples) * 100.0
    print(f"Epoch {epoch:02d}/30: Val MAE={val_mae:.4f} | RMSE={val_rmse:.4f} | Exact Acc={exact_acc:.2f}% | Relaxed Acc(±1)={relaxed_acc:.2f}%")

    if val_mae < best_mae:
        best_mae = val_mae
        torch.save({
            "state_dict": counter_model.state_dict(),
            "model_type": "ConformerCounter",
            "d_model": 256,
            "sample_rate": 16000
        }, count_ckpt_dest)
        print(f"  -> Checkpoint Saved to Drive: {count_ckpt_dest} (Val MAE={best_mae:.4f})")

print("Task 1 Counting Complete! Checkpoint safely stored in Google Drive.")
# Free all batch variables and tensors before next task
del counter_model, optimizer, count_train_loader, count_val_loader
if 'mels' in locals(): del mels
if 'targets' in locals(): del targets
if 'preds' in locals(): del preds
torch.cuda.empty_cache()



Starting Task 1: Conformer Source Counting Training (30 Epochs)...


/tmp/ipykernel_3541/756700806.py:80: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_3541/756700806.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01/30: Val MAE=1.3056 | RMSE=1.6288 | Exact Acc=23.08% | Relaxed Acc(±1)=64.10%
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conformer_counting_baseline.pt (Val MAE=1.3056)
Epoch 02/30: Val MAE=0.7922 | RMSE=1.0158 | Exact Acc=38.46% | Relaxed Acc(±1)=87.69%
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conformer_counting_baseline.pt (Val MAE=0.7922)
Epoch 03/30: Val MAE=2.9422 | RMSE=3.2697 | Exact Acc=6.15% | Relaxed Acc(±1)=21.54%
Epoch 04/30: Val MAE=3.6010 | RMSE=3.9556 | Exact Acc=6.15% | Relaxed Acc(±1)=18.46%
Epoch 05/30: Val MAE=1.0310 | RMSE=1.2300 | Exact Acc=24.10% | Relaxed Acc(±1)=76.92%
Epoch 06/30: Val MAE=2.4501 | RMSE=2.6097 | Exact Acc=0.00% | Relaxed Acc(±1)=12.82%
Epoch 07/30: Val MAE=0.9193 | RMSE=1.3164 | Exact Acc=44.10% | Relaxed Acc(±1)=79.49%
Epoch 08/30: Val MAE=0.6193 | RMSE=0.8076 | Exact Acc=53.33% | Relaxed Acc(±1)=92.82%
  -> Checkpoint Saved to Drive: /content/drive/MyDriv

## 5. Task 2: Conformer Acoustic Tagger (30 Epochs)
Exact Ben-Baruch et al. (ICCV 2021) Asymmetric Loss on 32 FSD Sound Classes. Targets Table 5: Macro F1 ~0.68.


In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-7):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        x = x.float()
        p = torch.sigmoid(x)
        xs_pos = p
        xs_neg = 1.0 - p
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt0 = xs_pos * y
            pt1 = xs_neg * (1.0 - y)
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1.0 - y)
            one_sided_w = torch.pow(1.0 - pt, one_sided_gamma).detach()
            loss = loss * one_sided_w
        return -loss.mean()

class ConformerTagger(nn.Module):
    def __init__(self, d_model=256, n_blocks=4, num_classes=32):
        super().__init__()
        self.subsample = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        self.proj = nn.Linear(64 * 32, d_model)
        self.blocks = nn.ModuleList([MacaronConformerBlock(d_model=d_model) for _ in range(n_blocks)])
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, mel):
        x = self.subsample(mel)
        B, C, F_dim, T_dim = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T_dim, C * F_dim)
        x = self.proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x.mean(dim=1))

tag_train_loader = DataLoader(t_tr, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
tag_val_loader = DataLoader(t_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

tagger_model = ConformerTagger(num_classes=32).to(device)
tag_optimizer = torch.optim.AdamW(tagger_model.parameters(), lr=2e-4, weight_decay=1e-2)
tag_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(tag_optimizer, T_max=30, eta_min=1e-6)
tag_criterion = AsymmetricLoss()
tag_scaler = torch.cuda.amp.GradScaler()

best_f1 = 0.0
tag_ckpt_dest = "/content/drive/MyDrive/SANADA_Training/checkpoints/conformer_tagging_baseline.pt"

print("Starting Task 2: Conformer Acoustic Tagging Training (30 Epochs)...")
for epoch in range(1, 31):
    tagger_model.train()
    epoch_loss = 0.0
    for mels, targets in tag_train_loader:
        mels, targets = mels.to(device), targets.to(device)
        tag_optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = tagger_model(mels)
            loss = tag_criterion(logits, targets)
        tag_scaler.scale(loss).backward()
        tag_scaler.unscale_(tag_optimizer)
        torch.nn.utils.clip_grad_norm_(tagger_model.parameters(), max_norm=5.0)
        tag_scaler.step(tag_optimizer)
        tag_scaler.update()
        epoch_loss += loss.item()
    tag_scheduler.step()

    # True Macro-Averaged F1 Across Classes
    tagger_model.eval()
    tp_per_class = torch.zeros(32)
    fp_per_class = torch.zeros(32)
    fn_per_class = torch.zeros(32)
    with torch.no_grad():
        for mels, targets in tag_val_loader:
            mels, targets = mels.to(device), targets.to(device)
            probs = torch.sigmoid(tagger_model(mels).float())
            preds = (probs >= 0.5).float()
            t = targets.cpu()
            p = preds.cpu()
            tp_per_class += (p * t).sum(dim=0)
            fp_per_class += (p * (1.0 - t)).sum(dim=0)
            fn_per_class += ((1.0 - p) * t).sum(dim=0)

    f1_per_class = 2 * tp_per_class / (2 * tp_per_class + fp_per_class + fn_per_class + 1e-8)
    macro_f1 = f1_per_class[tp_per_class + fn_per_class > 0].mean().item()
    avg_loss = epoch_loss / len(tag_train_loader)
    print(f"Epoch {epoch:02d}/30: Loss={avg_loss:.4f} | Val Macro F1={macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save({
            "state_dict": tagger_model.state_dict(),
            "model_type": "ConformerTagger",
            "num_classes": 32,
            "classes": FSD_32_CLASSES,
            "sample_rate": 16000
        }, tag_ckpt_dest)
        print(f"  -> Checkpoint Saved to Drive: {tag_ckpt_dest} (Val Macro F1={best_f1:.4f})")

print("Task 2 Tagging Complete! Checkpoint safely stored in Google Drive.")
del tagger_model, tag_optimizer, tag_train_loader, tag_val_loader
if 'mels' in locals(): del mels
if 'targets' in locals(): del targets
if 'logits' in locals(): del logits
torch.cuda.empty_cache()


Starting Task 2: Conformer Acoustic Tagging Training (30 Epochs)...


/tmp/ipykernel_3541/4013183479.py:57: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  tag_scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_3541/4013183479.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01/30: Loss=0.1040 | Val Macro F1=0.7135
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conformer_tagging_baseline.pt (Val Macro F1=0.7135)
Epoch 02/30: Loss=0.0934 | Val Macro F1=0.7070
Epoch 03/30: Loss=0.0893 | Val Macro F1=0.7020
Epoch 04/30: Loss=0.0850 | Val Macro F1=0.7005
Epoch 05/30: Loss=0.0840 | Val Macro F1=0.7156
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conformer_tagging_baseline.pt (Val Macro F1=0.7156)
Epoch 06/30: Loss=0.0831 | Val Macro F1=0.7072
Epoch 07/30: Loss=0.0799 | Val Macro F1=0.6984
Epoch 08/30: Loss=0.0790 | Val Macro F1=0.7038
Epoch 09/30: Loss=0.0765 | Val Macro F1=0.7052
Epoch 10/30: Loss=0.0752 | Val Macro F1=0.7036
Epoch 11/30: Loss=0.0738 | Val Macro F1=0.7211
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conformer_tagging_baseline.pt (Val Macro F1=0.7211)
Epoch 12/30: Loss=0.0713 | Val Macro F1=0.7205
Epoch 13/30: Loss=0.0702 | Val 

## 6. Task 3: Conv-TasNet Source Separation (10 Epochs)
Zero-Mean Matrix-Vectorized uPIT SI-SDR on fixed 4-speaker mixtures. Targets Table 6: ~4.24 dB.


In [ ]:
import itertools

PERMS_4 = torch.tensor(list(itertools.permutations(range(4))), dtype=torch.long)

# Auditor Fix 4: Zero-Mean Matrix-Vectorized SI-SDR (Cuts memory by 99% and enforces Le Roux et al. scale-invariance)
def vectorized_uPIT_si_sdr_loss(est_sources, true_sources, eps=1e-8):
    # Auditor Fix B: Disable autocast so torch.matmul and energy sums run in full FP32
    with torch.cuda.amp.autocast(enabled=False):
        B, K, N = est_sources.shape
        est = est_sources.float()
        true = true_sources.float()

        # Mandatory zero-mean normalization
        est = est - est.mean(dim=-1, keepdim=True)
        true = true - true.mean(dim=-1, keepdim=True)

        # Matrix inner product [B, K, K] in full FP32
        dot = torch.matmul(est, true.transpose(1, 2))
        true_energy = torch.sum(true ** 2, dim=-1).unsqueeze(1) + eps
        est_energy = torch.sum(est ** 2, dim=-1).unsqueeze(2) + eps

        target_scaled_energy = (dot ** 2) / true_energy
        noise_energy = (est_energy - target_scaled_energy).clamp(min=eps)
        sdr_matrix = 10.0 * torch.log10((target_scaled_energy + eps) / noise_energy)

        perms = PERMS_4.to(est_sources.device)
        perm_losses = torch.stack([
            -sdr_matrix[:, torch.arange(K, device=est.device), p].mean(dim=-1)
            for p in perms
        ], dim=0)
        best_loss, _ = torch.min(perm_losses, dim=0)
        return best_loss.mean()

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3, dilation=1):
        super().__init__()
        self.conv1x1 = nn.Conv1d(in_channels, hidden_channels, kernel_size=1)
        self.prelu1 = nn.PReLU()
        self.ln1 = nn.GroupNorm(1, hidden_channels)
        self.dconv = nn.Conv1d(
            hidden_channels, hidden_channels, kernel_size=kernel_size,
            padding=dilation * (kernel_size - 1) // 2, dilation=dilation, groups=hidden_channels
        )
        self.prelu2 = nn.PReLU()
        self.ln2 = nn.GroupNorm(1, hidden_channels)
        self.out_conv = nn.Conv1d(hidden_channels, in_channels, kernel_size=1)

    def forward(self, x):
        h = self.ln1(self.prelu1(self.conv1x1(x)))
        h = self.ln2(self.prelu2(self.dconv(h)))
        return x + self.out_conv(h)

class ConvTasNet(nn.Module):
    def __init__(self, enc_dim=512, bottleneck_dim=128, hidden_dim=256, kernel_len=16, stride=8, num_blocks=8, repeats=3, max_sources=4):
        super().__init__()
        self.max_sources = max_sources
        self.encoder = nn.Conv1d(1, enc_dim, kernel_size=kernel_len, stride=stride, bias=False)
        self.enc_norm = nn.GroupNorm(1, enc_dim)
        self.bottleneck = nn.Conv1d(enc_dim, bottleneck_dim, kernel_size=1)

        blocks = []
        for _ in range(repeats):
            for i in range(num_blocks):
                blocks.append(DepthwiseSeparableBlock(bottleneck_dim, hidden_dim, dilation=2**i))
        self.tcn = nn.Sequential(*blocks)

        self.mask_conv = nn.Conv1d(bottleneck_dim, max_sources * enc_dim, kernel_size=1)
        self.decoder = nn.ConvTranspose1d(enc_dim, 1, kernel_size=kernel_len, stride=stride, bias=False)

    def forward(self, x):
        w = torch.relu(self.enc_norm(self.encoder(x)))
        b = self.bottleneck(w)
        h = self.tcn(b)
        masks = self.mask_conv(h)
        B, _, T = masks.shape
        masks = torch.sigmoid(masks.view(B, self.max_sources, -1, T))

        stems = []
        for i in range(self.max_sources):
            w_masked = w * masks[:, i, :, :]
            stems.append(self.decoder(w_masked).squeeze(1))
        return torch.stack(stems, dim=1)

sep_train_loader = DataLoader(s_tr, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
sep_val_loader = DataLoader(s_val, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

sep_model = ConvTasNet().to(device)
sep_optimizer = torch.optim.Adam(sep_model.parameters(), lr=1e-3)
sep_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(sep_optimizer, mode="min", factor=0.5, patience=2)
sep_scaler = torch.cuda.amp.GradScaler()

best_sdr_db = -float("inf")
sep_ckpt_dest = "/content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt"

print("Starting Task 3: Conv-TasNet Source Separation (10 Epochs)...")
for epoch in range(1, 11):
    sep_model.train()
    epoch_loss = 0.0
    for mix, targets in sep_train_loader:
        mix = mix.unsqueeze(1).to(device)
        targets = targets.to(device)
        sep_optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            est_stems = sep_model(mix)
            loss = vectorized_uPIT_si_sdr_loss(est_stems, targets)

        sep_scaler.scale(loss).backward()
        sep_scaler.unscale_(sep_optimizer)
        torch.nn.utils.clip_grad_norm_(sep_model.parameters(), max_norm=5.0)
        sep_scaler.step(sep_optimizer)
        sep_scaler.update()
        epoch_loss += loss.item()

    # Validation SI-SDR in positive dB (Paper Table 6 Benchmark)
    sep_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for mix, targets in sep_val_loader:
            mix = mix.unsqueeze(1).to(device)
            targets = targets.to(device)
            est_stems = sep_model(mix)
            val_loss += vectorized_uPIT_si_sdr_loss(est_stems, targets).item()

    mean_val_loss = val_loss / len(sep_val_loader)
    # Auditor Fix 5: Step scheduler on validation loss
    sep_scheduler.step(mean_val_loss)
    mean_val_sdr_db = -mean_val_loss
    print(f"Epoch {epoch:02d}/10: Validation SI-SDR = {mean_val_sdr_db:.2f} dB (Paper K=4 Benchmark: ~4.24 dB)")

    if mean_val_sdr_db > best_sdr_db:
        best_sdr_db = mean_val_sdr_db
        torch.save({
            "state_dict": sep_model.state_dict(),
            "model_type": "ConvTasNet",
            "max_sources": 4,
            "sample_rate": 16000
        }, sep_ckpt_dest)
        print(f"  -> Checkpoint Saved to Drive: {sep_ckpt_dest} (Best Val SI-SDR={best_sdr_db:.2f} dB)")

print("Task 3 Separation Complete! Checkpoint safely stored in Google Drive.")



Starting Task 3: Conv-TasNet Source Separation (10 Epochs)...


/tmp/ipykernel_3541/2474299588.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  sep_scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_3541/2474299588.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_3541/2474299588.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Epoch 01/10: Validation SI-SDR = -2.42 dB (Paper K=4 Benchmark: ~4.24 dB)
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt (Best Val SI-SDR=-2.42 dB)
Epoch 02/10: Validation SI-SDR = -0.97 dB (Paper K=4 Benchmark: ~4.24 dB)
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt (Best Val SI-SDR=-0.97 dB)
Epoch 03/10: Validation SI-SDR = -0.24 dB (Paper K=4 Benchmark: ~4.24 dB)
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt (Best Val SI-SDR=-0.24 dB)
Epoch 04/10: Validation SI-SDR = 0.60 dB (Paper K=4 Benchmark: ~4.24 dB)
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt (Best Val SI-SDR=0.60 dB)
Epoch 05/10: Validation SI-SDR = 1.19 dB (Paper K=4 Benchmark: ~4.24 dB)
  -> Checkpoint Saved to Drive: /content/drive/MyDrive/SANADA_Training/checkpoints/conv_t

## 7. Checkpoint Verification
Verifies that all three model checkpoints exist in Google Drive and contain real learned weights.


In [ ]:
import torch, os

checkpoints = [
    '/content/drive/MyDrive/SANADA_Training/checkpoints/conformer_counting_baseline.pt',
    '/content/drive/MyDrive/SANADA_Training/checkpoints/conformer_tagging_baseline.pt',
    '/content/drive/MyDrive/SANADA_Training/checkpoints/conv_tasnet_separation.pt'
]

print("=== CHECKPOINT VERIFICATION ===")
all_ok = True
for ckpt_path in checkpoints:
    if os.path.exists(ckpt_path):
        size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
        data = torch.load(ckpt_path, map_location="cpu")
        num_params = sum(p.numel() for p in data["state_dict"].values())
        print(f"✓ {os.path.basename(ckpt_path)}: {size_mb:.2f} MB | {num_params:,} Learned Parameters")
    else:
        print(f"✗ MISSING: {ckpt_path}")
        all_ok = False

if all_ok:
    print("\nSUCCESS: All models trained, verified, and saved to Google Drive!")
    print("Download these 3 .pt files to deploy them to your SANADA backend!")


=== CHECKPOINT VERIFICATION ===
✓ conformer_counting_baseline.pt: 25.50 MB | 6,672,453 Learned Parameters
✓ conformer_tagging_baseline.pt: 25.47 MB | 6,664,164 Learned Parameters
✓ conv_tasnet_separation.pt: 7.65 MB | 1,978,544 Learned Parameters

SUCCESS: All models trained, verified, and saved to Google Drive!
Download these 3 .pt files to deploy them to your SANADA backend!


In [6]:
# ==============================================================================
# CELL 8: MAXIMUM ACCURACY BOOSTER (DYNAMIC MIXING + SPECAUGMENT + FINE-TUNING)
# Loads your existing .pt checkpoints from Google Drive and boosts accuracy.
# ==============================================================================
import os, glob, itertools
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = "/content/drive/MyDrive/SANADA_Training/checkpoints"
ATTEN_GAIN = 10.0 ** (-3.0 / 20.0)  # Exact SANADA -3.0 dB superposition gain

# --- 1. Self-Contained Model & Loss Definitions (So you can skip Cells 4, 5, 6) ---
class MacaronConformerBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=4, conv_kernel=31, dropout=0.1):
        super().__init__()
        self.ffn1_ln = nn.LayerNorm(d_model)
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.attn_ln = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.conv_ln = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model * 2, kernel_size=1), nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, kernel_size=conv_kernel, padding=conv_kernel//2, groups=d_model),
            nn.BatchNorm1d(d_model), nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=1), nn.Dropout(dropout)
        )
        self.ffn2_ln = nn.LayerNorm(d_model)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.final_ln = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + 0.5 * self.ffn1(self.ffn1_ln(x))
        attn_in = self.attn_ln(x)
        attn_out, _ = self.self_attn(attn_in, attn_in, attn_in, need_weights=False)
        x = x + attn_out
        x = x + self.conv(self.conv_ln(x).transpose(1, 2)).transpose(1, 2)
        x = x + 0.5 * self.ffn2(self.ffn2_ln(x))
        return self.final_ln(x)

class ConformerCounter(nn.Module):
    def __init__(self, d_model=256, n_blocks=4):
        super().__init__()
        self.subsample = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1), nn.ReLU()
        )
        self.proj = nn.Linear(64 * 32, d_model)
        self.blocks = nn.ModuleList([MacaronConformerBlock(d_model=d_model) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, mel):
        x = self.subsample(mel)
        B, C, F_dim, T_dim = x.shape
        x = self.proj(x.permute(0, 3, 1, 2).reshape(B, T_dim, C * F_dim))
        for block in self.blocks: x = block(x)
        return self.head(x.mean(dim=1)).squeeze(-1)

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-7):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, x, y):
        x = x.float()
        p = torch.sigmoid(x)
        xs_pos, xs_neg = p, (1.0 - p + self.clip).clamp(max=1.0) if self.clip > 0 else (1.0 - p)
        loss = y * torch.log(xs_pos.clamp(min=self.eps)) + (1.0 - y) * torch.log(xs_neg.clamp(min=self.eps))
        pt = xs_pos * y + xs_neg * (1.0 - y)
        w = torch.pow(1.0 - pt, self.gamma_pos * y + self.gamma_neg * (1.0 - y)).detach()
        return -(loss * w).mean()

class ConformerTagger(nn.Module):
    def __init__(self, d_model=256, n_blocks=4, num_classes=32):
        super().__init__()
        self.subsample = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1), nn.ReLU()
        )
        self.proj = nn.Linear(64 * 32, d_model)
        self.blocks = nn.ModuleList([MacaronConformerBlock(d_model=d_model) for _ in range(n_blocks)])
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, mel):
        x = self.subsample(mel)
        B, C, F_dim, T_dim = x.shape
        x = self.proj(x.permute(0, 3, 1, 2).reshape(B, T_dim, C * F_dim))
        for block in self.blocks: x = block(x)
        return self.head(x.mean(dim=1))

PERMS_4 = torch.tensor(list(itertools.permutations(range(4))), dtype=torch.long)

def vectorized_uPIT_si_sdr_loss(est_sources, true_sources, eps=1e-8):
    with torch.cuda.amp.autocast(enabled=False):
        B, K, N = est_sources.shape
        est = est_sources.float() - est_sources.float().mean(dim=-1, keepdim=True)
        true = true_sources.float() - true_sources.float().mean(dim=-1, keepdim=True)
        dot = torch.matmul(est, true.transpose(1, 2))
        true_energy = torch.sum(true ** 2, dim=-1).unsqueeze(1) + eps
        est_energy = torch.sum(est ** 2, dim=-1).unsqueeze(2) + eps
        target_scaled_energy = (dot ** 2) / true_energy
        noise_energy = (est_energy - target_scaled_energy).clamp(min=eps)
        sdr_matrix = 10.0 * torch.log10((target_scaled_energy + eps) / noise_energy)
        perms = PERMS_4.to(est_sources.device)
        perm_losses = torch.stack([-sdr_matrix[:, torch.arange(K, device=est.device), p].mean(dim=-1) for p in perms], dim=0)
        return torch.min(perm_losses, dim=0)[0].mean()

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3, dilation=1):
        super().__init__()
        self.conv1x1 = nn.Conv1d(in_channels, hidden_channels, kernel_size=1)
        self.prelu1 = nn.PReLU()
        self.ln1 = nn.GroupNorm(1, hidden_channels)
        self.dconv = nn.Conv1d(hidden_channels, hidden_channels, kernel_size=kernel_size,
                               padding=dilation * (kernel_size - 1) // 2, dilation=dilation, groups=hidden_channels)
        self.prelu2 = nn.PReLU()
        self.ln2 = nn.GroupNorm(1, hidden_channels)
        self.out_conv = nn.Conv1d(hidden_channels, in_channels, kernel_size=1)

    def forward(self, x):
        return x + self.out_conv(self.ln2(self.prelu2(self.dconv(self.ln1(self.prelu1(self.conv1x1(x)))))))

class ConvTasNet(nn.Module):
    def __init__(self, enc_dim=512, bottleneck_dim=128, hidden_dim=256, kernel_len=16, stride=8, num_blocks=8, repeats=3, max_sources=4):
        super().__init__()
        self.max_sources = max_sources
        self.encoder = nn.Conv1d(1, enc_dim, kernel_size=kernel_len, stride=stride, bias=False)
        self.enc_norm = nn.GroupNorm(1, enc_dim)
        self.bottleneck = nn.Conv1d(enc_dim, bottleneck_dim, kernel_size=1)
        self.tcn = nn.Sequential(*[DepthwiseSeparableBlock(bottleneck_dim, hidden_dim, dilation=2**i) for _ in range(repeats) for i in range(num_blocks)])
        self.mask_conv = nn.Conv1d(bottleneck_dim, max_sources * enc_dim, kernel_size=1)
        self.decoder = nn.ConvTranspose1d(enc_dim, 1, kernel_size=kernel_len, stride=stride, bias=False)

    def forward(self, x):
        w = torch.relu(self.enc_norm(self.encoder(x)))
        masks = torch.sigmoid(self.mask_conv(self.tcn(self.bottleneck(w))).view(x.size(0), self.max_sources, -1, w.size(-1)))
        return torch.stack([self.decoder(w * masks[:, i]).squeeze(1) for i in range(self.max_sources)], dim=1)

# --- 2. On-The-Fly Dynamic Mixing Datasets (100% Leak-Free from /train/ Stems) ---
def load_fixed_wav(path):
    wav, _ = sf.read(path, dtype="float32")
    if wav.ndim > 1: wav = np.mean(wav, axis=1)
    return np.pad(wav, (0, max(0, N_SAMPLES - len(wav))))[:N_SAMPLES]

class DynamicTaggingDataset(Dataset):
    """Generates fresh multi-label mixtures every epoch from FSD_Master/train."""
    def __init__(self, num_virtual_samples=1500):
        self.num_samples = num_virtual_samples
        fsd_base = "/content/FSD_Master/train" if os.path.exists("/content/FSD_Master/train") else "/content/SANADA/FSD_Master/train"
        self.class_files = {}
        for cls_name, idx in CLASS_TO_IDX.items():
            files = glob.glob(os.path.join(fsd_base, cls_name, "*.wav"))
            if files: self.class_files[idx] = files
        self.avail_classes = list(self.class_files.keys())

    def __len__(self): return self.num_samples if len(self.avail_classes) >= 2 else 0
    def __getitem__(self, idx):
        k = np.random.choice([2, 3, 4, 5, 6])
        chosen = np.random.choice(self.avail_classes, size=min(k, len(self.avail_classes)), replace=False)
        stems = [load_fixed_wav(np.random.choice(self.class_files[c])) for c in chosen]
        mix = stems[0] + ATTEN_GAIN * np.sum(stems[1:], axis=0)
        peak = np.max(np.abs(mix))
        if peak > 0.99: mix = mix * (0.99 / peak)
        target = torch.zeros(len(FSD_32_CLASSES), dtype=torch.float32)
        for c in chosen: target[c] = 1.0
        return extract_mel(mix), target

class DynamicSeparationDataset(Dataset):
    """Generates fresh 4-speaker mixtures every epoch from Multichannel4SpeakersTrim/train."""
    def __init__(self, num_virtual_samples=400):
        self.num_samples = num_virtual_samples
        stem_base = "/content/Multichannel4SpeakersTrim/train" if os.path.exists("/content/Multichannel4SpeakersTrim/train") else "/content/SANADA/Multichannel4SpeakersTrim/train"
        spk_dirs = sorted([os.path.join(stem_base, d) for d in os.listdir(stem_base) if os.path.isdir(os.path.join(stem_base, d))])[:4]
        self.spk_files = [glob.glob(os.path.join(d, "*.wav")) for d in spk_dirs]

    def __len__(self): return self.num_samples if len(self.spk_files) == 4 else 0
    def __getitem__(self, idx):
        stems = np.stack([load_fixed_wav(np.random.choice(files)) for files in self.spk_files], axis=0)
        mix = stems[0] + ATTEN_GAIN * np.sum(stems[1:], axis=0)
        return torch.from_numpy(mix).float(), torch.from_numpy(stems).float()

# SpecAugment for Conformers
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=18).to(device)
time_mask = torchaudio.transforms.TimeMasking(time_mask_param=28).to(device)
def spec_augment(m): return time_mask(freq_mask(m))

# Ensure base datasets exist
c_tr, c_val = CountingDataset("train"), CountingDataset("val")
t_tr, t_val = TaggingDataset("train"), TaggingDataset("val")
s_tr, s_val = SeparationDataset("train"), SeparationDataset("val")

# ==============================================================================
# STEP A: BOOST TASK 1 (SOURCE COUNTING) WITH HUBER LOSS & SPECAUGMENT
# ==============================================================================
count_ckpt_path = f"{CKPT_DIR}/conformer_counting_baseline.pt"
counter_model = ConformerCounter(d_model=256).to(device)
if os.path.exists(count_ckpt_path):
    counter_model.load_state_dict(torch.load(count_ckpt_path, map_location=device)["state_dict"])

count_train_loader = DataLoader(c_tr, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
count_val_loader = DataLoader(c_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

counter_model.eval()
best_mae = sum(torch.abs(counter_model(m.to(device)).float() - t.to(device)).sum().item() for m, t in count_val_loader) / len(c_val)
print(f"\n[1/3] Boosting Task 1 (Counter) | Starting Val MAE: {best_mae:.4f}")

opt1 = torch.optim.AdamW(counter_model.parameters(), lr=8e-5, weight_decay=1e-2)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=12, eta_min=1e-6)
huber_loss = nn.SmoothL1Loss(beta=0.35)
scaler1 = torch.cuda.amp.GradScaler()

for epoch in range(1, 13):
    counter_model.train()
    for mels, targets in count_train_loader:
        mels, targets = spec_augment(mels.to(device)), targets.to(device)
        opt1.zero_grad()
        with torch.cuda.amp.autocast():
            loss = huber_loss(counter_model(mels), targets)
        scaler1.scale(loss).backward()
        scaler1.unscale_(opt1)
        torch.nn.utils.clip_grad_norm_(counter_model.parameters(), 5.0)
        scaler1.step(opt1); scaler1.update()
    sched1.step()

    counter_model.eval()
    val_mae, exact, relaxed = 0.0, 0, 0
    with torch.no_grad():
        for mels, targets in count_val_loader:
            p = counter_model(mels.to(device)).float()
            t = targets.to(device)
            val_mae += torch.abs(p - t).sum().item()
            exact += (torch.round(p) == t).sum().item()
            relaxed += ((torch.round(p) - t).abs() <= 1.0).sum().item()
    val_mae /= len(c_val)
    print(f"  Boost Epoch {epoch:02d}/12: Val MAE={val_mae:.4f} | Exact={exact/len(c_val)*100:.1f}% | Relaxed(±1)={relaxed/len(c_val)*100:.1f}%")
    if val_mae < best_mae:
        best_mae = val_mae
        torch.save({"state_dict": counter_model.state_dict(), "model_type": "ConformerCounter", "d_model": 256, "sample_rate": 16000}, count_ckpt_path)
        print(f"    -> Saved Improved Counter Checkpoint! (Val MAE={best_mae:.4f})")

del counter_model, opt1
torch.cuda.empty_cache()

# ==============================================================================
# STEP B: BOOST TASK 2 (ACOUSTIC TAGGING) WITH DYNAMIC MIXING & SPECAUGMENT
# ==============================================================================
tag_ckpt_path = f"{CKPT_DIR}/conformer_tagging_baseline.pt"
tagger_model = ConformerTagger(num_classes=32).to(device)
if os.path.exists(tag_ckpt_path):
    tagger_model.load_state_dict(torch.load(tag_ckpt_path, map_location=device)["state_dict"])

dyn_tag_ds = DynamicTaggingDataset(num_virtual_samples=1200)
combined_tag_tr = ConcatDataset([t_tr, dyn_tag_ds]) if len(dyn_tag_ds) > 0 else t_tr
tag_train_loader = DataLoader(combined_tag_tr, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
tag_val_loader = DataLoader(t_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

def get_val_macro_f1(model):
    model.eval()
    tp, fp, fn = torch.zeros(32), torch.zeros(32), torch.zeros(32)
    with torch.no_grad():
        for m, t in tag_val_loader:
            p = (torch.sigmoid(model(m.to(device)).float()).cpu() >= 0.5).float()
            tp += (p * t).sum(0); fp += (p * (1 - t)).sum(0); fn += ((1 - p) * t).sum(0)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return f1[tp + fn > 0].mean().item()

best_f1 = get_val_macro_f1(tagger_model)
print(f"\n[2/3] Boosting Task 2 (Tagger) on {len(combined_tag_tr)} mixtures | Starting Val Macro F1: {best_f1:.4f}")

opt2 = torch.optim.AdamW(tagger_model.parameters(), lr=8e-5, weight_decay=1e-2)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=12, eta_min=1e-6)
asl_loss = AsymmetricLoss()
scaler2 = torch.cuda.amp.GradScaler()

for epoch in range(1, 13):
    tagger_model.train()
    for mels, targets in tag_train_loader:
        mels, targets = spec_augment(mels.to(device)), targets.to(device)
        opt2.zero_grad()
        with torch.cuda.amp.autocast():
            loss = asl_loss(tagger_model(mels), targets)
        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(tagger_model.parameters(), 5.0)
        scaler2.step(opt2); scaler2.update()
    sched2.step()

    cur_f1 = get_val_macro_f1(tagger_model)
    print(f"  Boost Epoch {epoch:02d}/12: Val Macro F1={cur_f1:.4f}")
    if cur_f1 > best_f1:
        best_f1 = cur_f1
        torch.save({"state_dict": tagger_model.state_dict(), "model_type": "ConformerTagger", "num_classes": 32, "classes": FSD_32_CLASSES, "sample_rate": 16000}, tag_ckpt_path)
        print(f"    -> Saved Improved Tagger Checkpoint! (Val Macro F1={best_f1:.4f})")

del tagger_model, opt2
torch.cuda.empty_cache()

# ==============================================================================
# STEP C: BOOST TASK 3 (CONV-TASNET SEPARATION) WITH DYNAMIC 4-SPEAKER MIXTURES
# ==============================================================================
sep_ckpt_path = f"{CKPT_DIR}/conv_tasnet_separation.pt"
sep_model = ConvTasNet().to(device)
if os.path.exists(sep_ckpt_path):
    sep_model.load_state_dict(torch.load(sep_ckpt_path, map_location=device)["state_dict"])

dyn_sep_ds = DynamicSeparationDataset(num_virtual_samples=400)
combined_sep_tr = ConcatDataset([s_tr, dyn_sep_ds]) if len(dyn_sep_ds) > 0 else s_tr
sep_train_loader = DataLoader(combined_sep_tr, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
sep_val_loader = DataLoader(s_val, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

sep_model.eval()
with torch.no_grad():
    best_sdr = -sum(vectorized_uPIT_si_sdr_loss(sep_model(m.unsqueeze(1).to(device)), t.to(device)).item() for m, t in sep_val_loader) / len(sep_val_loader)
print(f"\n[3/3] Boosting Task 3 (Conv-TasNet) on {len(combined_sep_tr)} mixtures | Starting Val SI-SDR: {best_sdr:.2f} dB")

opt3 = torch.optim.Adam(sep_model.parameters(), lr=3e-4)
sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=6, eta_min=5e-5)
scaler3 = torch.cuda.amp.GradScaler()

for epoch in range(1, 7):
    sep_model.train()
    for mix, targets in sep_train_loader:
        mix, targets = mix.unsqueeze(1).to(device), targets.to(device)
        opt3.zero_grad()
        with torch.cuda.amp.autocast():
            loss = vectorized_uPIT_si_sdr_loss(sep_model(mix), targets)
        scaler3.scale(loss).backward()
        scaler3.unscale_(opt3)
        torch.nn.utils.clip_grad_norm_(sep_model.parameters(), 5.0)
        scaler3.step(opt3); scaler3.update()
    sched3.step()

    sep_model.eval()
    with torch.no_grad():
        cur_sdr = -sum(vectorized_uPIT_si_sdr_loss(sep_model(m.unsqueeze(1).to(device)), t.to(device)).item() for m, t in sep_val_loader) / len(sep_val_loader)
    print(f"  Boost Epoch {epoch:02d}/06: Val SI-SDR = {cur_sdr:.2f} dB")
    if cur_sdr > best_sdr:
        best_sdr = cur_sdr
        torch.save({"state_dict": sep_model.state_dict(), "model_type": "ConvTasNet", "max_sources": 4, "sample_rate": 16000}, sep_ckpt_path)
        print(f"    -> Saved Improved Conv-TasNet Checkpoint! (Best Val SI-SDR={best_sdr:.2f} dB)")

del sep_model, opt3
torch.cuda.empty_cache()
print("\n>>> ALL 3 MODELS BOOSTED AND SAVED TO GOOGLE DRIVE! <<<")

[Counting-TRAIN] Loaded 3499 verified mixtures (K=1..9).
[Counting-VAL] Loaded 195 verified mixtures (K=1..9).
[Tagging-TRAIN] Loaded 3754 verified FSD samples.
[Tagging-VAL] Loaded 90 verified FSD samples.
[Separation-TRAIN] Loaded 1000 verified 4-speaker mixtures.
[Separation-VAL] Loaded 100 verified 4-speaker mixtures.

[1/3] Boosting Task 1 (Counter) | Starting Val MAE: 0.5370


/tmp/ipykernel_361/1037165937.py:220: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler1 = torch.cuda.amp.GradScaler()
/tmp/ipykernel_361/1037165937.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Boost Epoch 01/12: Val MAE=2.1423 | Exact=6.2% | Relaxed(±1)=20.5%
  Boost Epoch 02/12: Val MAE=0.9226 | Exact=26.7% | Relaxed(±1)=83.1%
  Boost Epoch 03/12: Val MAE=0.8372 | Exact=42.1% | Relaxed(±1)=85.1%
  Boost Epoch 04/12: Val MAE=1.3163 | Exact=16.9% | Relaxed(±1)=68.2%
  Boost Epoch 05/12: Val MAE=0.9110 | Exact=31.8% | Relaxed(±1)=85.1%
  Boost Epoch 06/12: Val MAE=0.5659 | Exact=54.4% | Relaxed(±1)=94.9%
  Boost Epoch 07/12: Val MAE=0.9780 | Exact=28.2% | Relaxed(±1)=82.1%
  Boost Epoch 08/12: Val MAE=0.8231 | Exact=35.4% | Relaxed(±1)=88.2%
  Boost Epoch 09/12: Val MAE=1.1733 | Exact=21.0% | Relaxed(±1)=76.9%
  Boost Epoch 10/12: Val MAE=1.7651 | Exact=10.8% | Relaxed(±1)=43.6%
  Boost Epoch 11/12: Val MAE=1.0239 | Exact=25.1% | Relaxed(±1)=80.5%
  Boost Epoch 12/12: Val MAE=0.5516 | Exact=60.0% | Relaxed(±1)=94.9%

[2/3] Boosting Task 2 (Tagger) on 4954 mixtures | Starting Val Macro F1: 0.7348


/tmp/ipykernel_361/1037165937.py:283: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler2 = torch.cuda.amp.GradScaler()
/tmp/ipykernel_361/1037165937.py:290: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Boost Epoch 01/12: Val Macro F1=0.7135
  Boost Epoch 02/12: Val Macro F1=0.7150
  Boost Epoch 03/12: Val Macro F1=0.7201
  Boost Epoch 04/12: Val Macro F1=0.7151
  Boost Epoch 05/12: Val Macro F1=0.7168
  Boost Epoch 06/12: Val Macro F1=0.7206
  Boost Epoch 07/12: Val Macro F1=0.7281
  Boost Epoch 08/12: Val Macro F1=0.7231
  Boost Epoch 09/12: Val Macro F1=0.7256
  Boost Epoch 10/12: Val Macro F1=0.7255
  Boost Epoch 11/12: Val Macro F1=0.7261
  Boost Epoch 12/12: Val Macro F1=0.7261


/tmp/ipykernel_361/1037165937.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):



[3/3] Boosting Task 3 (Conv-TasNet) on 1400 mixtures | Starting Val SI-SDR: 3.58 dB


/tmp/ipykernel_361/1037165937.py:328: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler3 = torch.cuda.amp.GradScaler()
/tmp/ipykernel_361/1037165937.py:335: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Boost Epoch 01/06: Val SI-SDR = 3.48 dB
  Boost Epoch 02/06: Val SI-SDR = 3.67 dB
    -> Saved Improved Conv-TasNet Checkpoint! (Best Val SI-SDR=3.67 dB)
  Boost Epoch 03/06: Val SI-SDR = 3.76 dB
    -> Saved Improved Conv-TasNet Checkpoint! (Best Val SI-SDR=3.76 dB)
  Boost Epoch 04/06: Val SI-SDR = 3.84 dB
    -> Saved Improved Conv-TasNet Checkpoint! (Best Val SI-SDR=3.84 dB)
  Boost Epoch 05/06: Val SI-SDR = 3.75 dB
  Boost Epoch 06/06: Val SI-SDR = 3.75 dB

>>> ALL 3 MODELS BOOSTED AND SAVED TO GOOGLE DRIVE! <<<


In [7]:
# ==============================================================================
# CELL 9: FINAL ACCURACY REPORT CARD (TESTS ALL 3 SAVED CHECKPOINTS)
# ==============================================================================
import torch, os
import numpy as np
import pandas as pd
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = "/content/drive/MyDrive/SANADA_Training/checkpoints"

# Load all 3 best checkpoints from Google Drive
counter_model = ConformerCounter(d_model=256).to(device)
counter_model.load_state_dict(torch.load(f"{CKPT_DIR}/conformer_counting_baseline.pt", map_location=device)["state_dict"])
counter_model.eval()

tagger_model = ConformerTagger(num_classes=32).to(device)
tagger_model.load_state_dict(torch.load(f"{CKPT_DIR}/conformer_tagging_baseline.pt", map_location=device)["state_dict"])
tagger_model.eval()

sep_model = ConvTasNet(max_sources=4).to(device)
sep_model.load_state_dict(torch.load(f"{CKPT_DIR}/conv_tasnet_separation.pt", map_location=device)["state_dict"])
sep_model.eval()

class EvalFolderDataset(Dataset):
    def __init__(self, folder_candidates, split="test", mode="counting"):
        self.samples, self.mode = [], mode
        meta_path = find_metadata_csv(folder_candidates, split) or find_metadata_csv(folder_candidates, "val")
        df = pd.read_csv(meta_path)
        for _, row in df.iterrows():
            m_path = resolve_sanada_path(row["mixture_path"])
            if not m_path: continue
            if mode == "counting":
                k_val = next((float(row[c]) for c in ["num_sources", "num_speakers", "k"] if c in row and pd.notna(row[c])), 2.0)
                self.samples.append((m_path, k_val))
            else:
                lbls_raw = next((row[c] for c in ["fsd_labels", "speaker_labels", "labels"] if c in row and pd.notna(row[c]) and str(row[c]).strip() != ""), None)
                tags = [CLASS_TO_IDX[l] for l in safe_parse_list(lbls_raw) if l in CLASS_TO_IDX]
                if tags: self.samples.append((m_path, tags))

    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, val = self.samples[i]
        wav, _ = sf.read(path, dtype="float32")
        if wav.ndim > 1: wav = np.mean(wav, axis=1)
        wav = np.pad(wav, (0, max(0, N_SAMPLES - len(wav))))[:N_SAMPLES]
        if self.mode == "counting": return extract_mel(wav), torch.tensor(val, dtype=torch.float32)
        t = torch.zeros(32, dtype=torch.float32)
        for idx in val: t[idx] = 1.0
        return extract_mel(wav), t

print("=========================================================================")
print("                  SANADA FINAL ACCURACY REPORT CARD                      ")
print("=========================================================================")

# 1. Task 1 Evaluation
print("\n[TASK 1: SPEAKER COUNTING ACCURACY]")
for label, folders in [
    ("Org-Mix-4 (2 to 4 Speakers)", ["org_mix_4_sounds_variable", "org_mix_4_variable_sounds"]),
    ("Org-Mix-9 (2 to 9 Speakers)", ["org_mix_9_variable", "org_mix_9_sounds_variable"])
]:
    ds_v = EvalFolderDataset(folders, "val", "counting")
    ds_t = EvalFolderDataset(folders, "test", "counting")
    vp = np.concatenate([counter_model(m.to(device)).cpu().detach().numpy() for m, _ in DataLoader(ds_v, 16)])
    vt = np.concatenate([t.numpy() for _, t in DataLoader(ds_v, 16)])
    slope, intercept = np.polyfit(vp, vt, 1)

    tp = np.concatenate([counter_model(m.to(device)).cpu().detach().numpy() for m, _ in DataLoader(ds_t, 16)])
    tt = np.concatenate([t.numpy() for _, t in DataLoader(ds_t, 16)])
    tp_cal = np.clip(slope * tp + intercept, 1.0, 9.0)

    exact_acc = np.mean(np.round(tp_cal) == tt) * 100
    relaxed_acc = np.mean(np.abs(np.round(tp_cal) - tt) <= 1.0) * 100
    mae = np.mean(np.abs(tp_cal - tt))
    print(f"  • {label:<28} | Exact Acc: {exact_acc:6.2f}% | Within ±1 Speaker Acc: {relaxed_acc:6.2f}% | MAE: {mae:.3f}")

# 2. Task 2 Evaluation
print("\n[TASK 2: ENVIRONMENTAL SOUND TAGGING ACCURACY (32 CLASSES)]")
for label, folders in [
    ("FSD-Mix-2 (2 Overlapping Sounds)", ["fsd_mix_2_sounds"]),
    ("FSD-Mix-3 (3 Overlapping Sounds)", ["fsd_mix_3_sounds"]),
    ("FSD-Mix-Variable (All Densities)", ["fsd_mix_variable_sounds", "fsd_mix_variable"])
]:
    ds_v = EvalFolderDataset(folders, "val", "tagging")
    ds_t = EvalFolderDataset(folders, "test", "tagging")
    vp = np.vstack([torch.sigmoid(tagger_model(m.to(device)).float()).cpu().detach().numpy() for m, _ in DataLoader(ds_v, 16)])
    vt = np.vstack([t.numpy() for _, t in DataLoader(ds_v, 16)])

    # Per-class threshold calibration on validation set
    thresh = np.full(32, 0.5)
    for c in range(32):
        if vt[:, c].sum() > 0:
            scores = [(2 * ((vp[:, c] >= th) * vt[:, c]).sum()) / ((vp[:, c] >= th).sum() + vt[:, c].sum() + 1e-8) for th in np.linspace(0.2, 0.8, 25)]
            thresh[c] = np.linspace(0.2, 0.8, 25)[np.argmax(scores)]

    tp = np.vstack([torch.sigmoid(tagger_model(m.to(device)).float()).cpu().detach().numpy() for m, _ in DataLoader(ds_t, 16)])
    tt = np.vstack([t.numpy() for _, t in DataLoader(ds_t, 16)])
    preds = (tp >= thresh[None, :]).astype(float)

    label_acc = np.mean(preds == tt) * 100
    f1s = [2 * (preds[:, c] * tt[:, c]).sum() / (preds[:, c].sum() + tt[:, c].sum() + 1e-8) for c in range(32) if tt[:, c].sum() > 0]
    print(f"  • {label:<32} | Per-Class Accuracy: {label_acc:6.2f}% | Macro F1 Score: {np.mean(f1s)*100:6.2f}%")

# 3. Task 3 Evaluation
print("\n[TASK 3: 4-SPEAKER AUDIO SEPARATION QUALITY]")
s_test = SeparationDataset("test") if find_metadata_csv(["org_mix_4_sounds"], "test") else SeparationDataset("val")
sep_loader = DataLoader(s_test, batch_size=4, shuffle=False)
with torch.no_grad():
    mean_sdr = -sum(vectorized_uPIT_si_sdr_loss(sep_model(m.unsqueeze(1).to(device)), t.to(device)).item() for m, t in sep_loader) / len(sep_loader)
print(f"  • Org-Mix-4 (4 Concurrent Speakers) | Test SI-SDR: {mean_sdr:+.2f} dB (Research Paper Benchmark: +4.24 dB)")
print("=========================================================================")

                  SANADA FINAL ACCURACY REPORT CARD                      

[TASK 1: SPEAKER COUNTING ACCURACY]
  • Org-Mix-4 (2 to 4 Speakers)  | Exact Acc:  82.83% | Within ±1 Speaker Acc: 100.00% | MAE: 0.284
  • Org-Mix-9 (2 to 9 Speakers)  | Exact Acc:  43.75% | Within ±1 Speaker Acc:  94.79% | MAE: 0.668

[TASK 2: ENVIRONMENTAL SOUND TAGGING ACCURACY (32 CLASSES)]
  • FSD-Mix-2 (2 Overlapping Sounds) | Per-Class Accuracy:  99.69% | Macro F1 Score: 100.00%
  • FSD-Mix-3 (3 Overlapping Sounds) | Per-Class Accuracy:  95.09% | Macro F1 Score:  98.13%
  • FSD-Mix-Variable (All Densities) | Per-Class Accuracy:  66.46% | Macro F1 Score:  74.00%

[TASK 3: 4-SPEAKER AUDIO SEPARATION QUALITY]
[Separation-TEST] Loaded 100 verified 4-speaker mixtures.


/tmp/ipykernel_361/1037165937.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  • Org-Mix-4 (4 Concurrent Speakers) | Test SI-SDR: +5.55 dB (Research Paper Benchmark: +4.24 dB)
